# Extract from Wikipedia and Fandom

## Fetch the contents

In [1]:
from pathlib import Path
import json

RAW_WIKI_DIR = Path("input/wiki")
OUTPUT_DIR = Path("output/wiki")
WIKI_PAGES = {
    "characters": "https://it.wikipedia.org/wiki/Personaggi_di_PK",
    "issues": "https://it.wikipedia.org/wiki/Albi_di_PK_-_Paperinik_New_Adventures",
    "quotes": "https://it.wikiquote.org/wiki/PK_-_Paperinik_New_Adventures",
    "lyla": "https://it.wikipedia.org/wiki/Lyla_Lay",
    "paperinik": "https://it.wikipedia.org/wiki/Paperinik",
    "uno": "https://it.wikipedia.org/wiki/Uno_(personaggio)",
}
FANDOM_PAGES = {
    "uno": "https://disney-comics.fandom.com/it/wiki/Uno",
    "paperinik": "https://disney-comics.fandom.com/it/wiki/Pikappa",
    "pkna": "https://disney-comics.fandom.com/it/wiki/Paperinik_New_Adventures",
    "citazioni-pk": "https://disney-comics.fandom.com/it/wiki/Citazioni_di_Pikappa",
    "citazioni-uno": "https://disney-comics.fandom.com/it/wiki/Citazioni_di_Uno",
}
FANDOM_CATEGORIES = [
    "Categoria:Personaggi di Pikappa",
    "Categoria:Luoghi di Pikappa",
    "Categoria:Storie di Pikappa",
    "Categoria:Tecnologia di Pikappa",
]

# Generate with the following prompt:
#
# Given the following HTML table, extract all the links present in the third column (Titolo), and issue number from the second column (Numero), in the following JSON format:
# list of tuples: issue number, title, url.
# <here the HTML table pasted from https://disney-comics.fandom.com/it/wiki/Paperinik_New_Adventures
FANDOM_ISSUES_PATH = Path("input/wiki/fandom/issues.json")
FANDOM_ISSUES = json.loads(FANDOM_ISSUES_PATH.read_text(encoding="utf-8"))

In [11]:
from pathlib import Path
import re


def name_to_id(name: str) -> str:
    """
    Convert a name to a valid ID by replacing non-alphanumeric characters with dashes,
    removing consecutive dashes, and converting to lowercase.
    """
    return re.sub(r"[^a-zA-Z0-9]", "-", name).replace("--", "-").strip("-").lower()


def name_to_path(name: str, prefix: str = "") -> Path:
    id = name_to_id(name)
    return RAW_WIKI_DIR / prefix / f"{id}.wiki"


def save_mediawiki(title: str, contents: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    buf = f"= {title} =\n\n" + contents + "\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(buf)

In [12]:
import requests
from tqdm.notebook import tqdm


def make_wiki_download_url(page_url: str) -> str:
    """Return the MediaWiki API URL for the given page URL using regex extraction."""
    pattern = re.compile(
        r"^https://(?P<country>[a-z]{2})\.(?P<host>[a-z]+\.org)/wiki/(?P<title>[^?#]+)$"
    )
    m = pattern.match(page_url)
    if not m:
        raise ValueError(f"Unsupported page URL format: {page_url}")
    country = m.group("country")
    host = m.group("host")
    title = m.group("title")  # already in underscore form
    api_url = (
        f"https://{country}.{host}/w/api.php?action=query&prop=revisions&titles={title}"
        f"&rvslots=main&rvprop=content&formatversion=2&format=json"
    )
    return api_url


def download_wiki_mediawiki(page_url: str) -> tuple[str, str]:
    """Download the content of a MediaWiki page and return its title and content."""
    api_url = make_wiki_download_url(page_url)
    response = requests.get(api_url)
    response.raise_for_status()
    data = response.json()
    title = data["query"]["pages"][0]["title"]
    contents = data["query"]["pages"][0]["revisions"][0]["slots"]["main"]["content"]
    return title, contents


for name, url in tqdm(WIKI_PAGES.items()):
    p = name_to_path(name)
    if p.exists():
        continue
    title, contents = download_wiki_mediawiki(url)
    save_mediawiki(title, contents, p)

  0%|          | 0/6 [00:00<?, ?it/s]

In [13]:
from tqdm.notebook import tqdm


def make_fandom_download_url(page_url: str) -> str:
    """Return the Fandom API URL for the given page URL using regex extraction."""
    pattern = re.compile(
        r"^https://disney-comics\.fandom\.com/it/wiki/(?P<title>[^?#]+)$"
    )
    m = pattern.match(page_url)
    if not m:
        raise ValueError(f"Unsupported page URL format: {page_url}")
    title = m.group("title")  # already in underscore form
    api_url = (
        f"https://disney-comics.fandom.com/it/api.php?action=query&titles={title}"
        f"&prop=revisions&rvprop=content&format=json"
    )
    return api_url


def download_wiki_mediawiki(page_url: str) -> tuple[str, str]:
    api_url = make_fandom_download_url(page_url)
    response = requests.get(api_url)
    response.raise_for_status()
    data = response.json()
    pages = data["query"]["pages"]
    if len(pages) != 1:
        raise ValueError(f"Expected exactly one page in response, got {len(pages)}")
    page = next(iter(pages.values()))
    title = page["title"]
    contents = page["revisions"][0]["*"]
    return title, contents


# Download main articles
for name, url in tqdm(FANDOM_PAGES.items(), desc="Main articles"):
    p = name_to_path(name, "fandom")
    if p.exists():
        continue
    title, contents = download_wiki_mediawiki(url)
    save_mediawiki(title, contents, p)

# Download issues
for name, title, url_path in tqdm(FANDOM_ISSUES, desc="Issues"):
    p = name_to_path(name, "fandom/issues")
    if p.exists():
        continue
    url = f"https://disney-comics.fandom.com{url_path}"
    title, contents = download_wiki_mediawiki(url)
    save_mediawiki(title, contents, p)

Main articles:   0%|          | 0/5 [00:00<?, ?it/s]

Issues:   0%|          | 0/56 [00:00<?, ?it/s]

In [ ]:
def crawl_fandom_category(name: str, prefix: str = "") -> list[tuple[str, str]]:
    """Recursively crawl a fandom category and return all member titles.

    Returns a list of tuples containing the sub-category and the title of each member.
    """
    url = (
        f"https://disney-comics.fandom.com/it/api.php"
        f"?action=query&list=categorymembers&cmtitle={name}"
        f"&cmlimit=max&format=json"
    )
    response = requests.get(url)
    data = response.json()
    category = name.replace("Categoria:", "").replace(" di Pikappa", "")
    category = str(Path(prefix) / name_to_id(category))
    members = data.get("query", {}).get("categorymembers", [])
    res = []

    for member in members:
        title = member["title"]
        if title.startswith("Categoria:"):
            # Recurse on sub-categories
            sub_results = crawl_fandom_category(title, prefix=category)
            res.extend(sub_results)
        else:
            res.append((category, title))

    return res

In [15]:
raw_fandom_pages: list[tuple[str, str]] = []
for category in FANDOM_CATEGORIES:
    pages = crawl_fandom_category(category, prefix="fandom/crawl")
    raw_fandom_pages.extend(pages)

In [17]:
from urllib.parse import quote

# Download all from crawl
for category, title in tqdm(raw_fandom_pages, desc="Crawled pages"):
    p = name_to_path(title, category)
    if p.exists():
        continue
    page = quote(title.replace(" ", "_"), safe="")
    url = f"https://disney-comics.fandom.com/it/wiki/{page}"
    title, contents = download_wiki_mediawiki(url)
    save_mediawiki(title, contents, p)

Crawled pages:   0%|          | 0/404 [00:00<?, ?it/s]

## Convert to Markdown

In [18]:
import shutil
import subprocess


def mediawiki_to_md(mediawiki_contents: str) -> str:
    pandoc = shutil.which("pandoc")
    if not pandoc:
        raise RuntimeError("pandoc is not installed")
    proc = subprocess.run(
        [
            pandoc,
            "--from=mediawiki",
            "--to=markdown_strict",
            "--wrap=preserve",
            "--strip-comments",
            "--lua-filter=pandoc-strip-fmt.lua",
        ],
        input=mediawiki_contents,
        text=True,
        capture_output=True,
        check=True,
    )
    txt = proc.stdout
    txt = txt.replace("{{'}}", "'")
    txt = txt.replace("<!-- -->\n", "")
    txt = txt.replace("\\#", "#")
    txt = re.sub(r"(?m)^\s*={4}\s*(.+?)\s*={4}\s*$", r"\n#### \1\n", txt)
    txt = re.sub(r"(?m)^\s*={3}\s*(.+?)\s*={3}\s*$", r"\n### \1\n", txt)
    txt = re.sub(r"(?m)^\s*={2}\s*(.+?)\s*={2}\s*$", r"\n## \1\n", txt)
    return txt

In [19]:
# Convert all wiki pages to MediaWiki format
from glob import glob

wiki_files = [
    Path(p) for p in glob(str(RAW_WIKI_DIR / "**" / "*.wiki"), recursive=True)
]

for wiki_file in tqdm(wiki_files, desc="Wiki files"):
    suffix = wiki_file.relative_to(RAW_WIKI_DIR).parent
    name = f"{wiki_file.stem}.md"
    output = OUTPUT_DIR / suffix / name
    output.parent.mkdir(parents=True, exist_ok=True)
    in_contents = wiki_file.read_text(encoding="utf-8")
    output.write_text(mediawiki_to_md(in_contents))

Wiki files:   0%|          | 0/472 [00:00<?, ?it/s]

In [20]:
def mediawiki_to_txt(mediawiki_contents: str) -> str:
    pandoc = shutil.which("pandoc")
    if not pandoc:
        raise RuntimeError("pandoc is not installed")
    proc = subprocess.run(
        [
            pandoc,
            "--from=mediawiki",
            "--to=plain",
            "--wrap=preserve",
            "--lua-filter=pandoc-split-h.lua",
        ],
        input=mediawiki_contents,
        text=True,
        capture_output=True,
        check=True,
    )
    return proc.stdout.replace("{{'}}", "'")


# for title, (_, content) in mediawiki_contents.items():
#     txt = mediawiki_to_txt(content)
#     output_file = OUTPUT_DIR / f"{title}.txt"
#     with open(output_file, "w", encoding="utf-8") as f:
#         f.write(txt)